In [8]:
import keras
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Ensures identical structural initializations
keras.utils.set_random_seed(42)

# Load Fashion-MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
_, counts = np.unique(y_train, return_counts=True)
print("Value counts of y_train:", counts)

# Normalize pixel values to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flatten 28 x 28 images into 784 features
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# Spliting training data into training and validation subsets.
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)

Value counts of y_train: [6000 6000 6000 6000 6000 6000 6000 6000 6000 6000]


# Task 1 – Build a baseline model
Build a baseline model with one hidden layer of 32 neurons using ReLU activation and the Adam optimizer. Train it and record its training and validation accuracy.

In [9]:
def create_and_fit_model(hidden_neurons: int, hidden_activation, optimizer: str) -> tuple[keras.Sequential, keras.callbacks.History]:
    model = keras.Sequential(
        [
            keras.Input(
                shape=(784,)
            ),  # One feature for each pixel in an 28 × 28 image.
            keras.layers.Dense(
                hidden_neurons, activation=hidden_activation,
            ),  # x hidden neurons; nonlinear transformations.
            keras.layers.Dense(
                10, activation="softmax"
            ),  # One output probability per fashion category, summing to 1.
        ]
    )

    model.compile(
        loss=keras.losses.sparse_categorical_crossentropy,  # What counts as error
        optimizer=optimizer,  # How weights update
        metrics=[
            keras.metrics.SparseCategoricalAccuracy()
        ],  # What performance to report
    )

    history = model.fit(
        x_train,
        y_train,
        epochs=10,
        batch_size=64,
        verbose=0,
        validation_data=(x_val, y_val),
    )

    return model, history

baseline, base_hist = create_and_fit_model(32, keras.activations.relu, keras.optimizers.Adam())

print(f"Baseline Training Accuracy: {base_hist.history['sparse_categorical_accuracy'][-1]:.4f}")
print(f"Baseline Validation Accuracy: {base_hist.history['val_sparse_categorical_accuracy'][-1]:.4f}")

Baseline Training Accuracy: 0.8897
Baseline Validation Accuracy: 0.8711


# Task 2 – Run experiments 
Starting from the baseline, run three more experiments. For each experiment, change only the setting shown below; keep the other settings the same as the baseline.

| Model | Hidden neurons | Hidden activation | Optimizer |
| --- | --- | --- | --- |
| Baseline | 32 | ReLU | Adam | Row 1 |
| Experiment 1 | 64 | ReLU | Adam |
| Experiment 2 | 32 | tanh | Adam |
| Experiment 3 | 32 | ReLU | SGD |

In [10]:
exp_1_model, exp_1_hist = create_and_fit_model(64, keras.activations.relu, keras.optimizers.Adam())
exp_2_model, exp_2_hist = create_and_fit_model(32, keras.activations.tanh, keras.optimizers.Adam())
exp_3_model, exp_3_hist = create_and_fit_model(32, keras.activations.relu, keras.optimizers.SGD())

# Task 3 – Record -> indentify -> evaluate
Record the training and validation accuracy of all four models in a results table. Identify the model with the highest validation accuracy. Evaluate that model on the test set and report its test accuracy. Do not use the test set to choose the model.

In [11]:
history_list = [base_hist, exp_1_hist, exp_2_hist, exp_3_hist]
model_names = ['Baseline', 'Experiment 1', 'Experiment 2', 'Experiment 3']

data = []
for name, hist in zip(model_names, history_list):
    h = hist.history 
    data.append({
        'Model': name,
        'Train Accuracy': h['sparse_categorical_accuracy'][-1],
        'Val Accuracy': h['val_sparse_categorical_accuracy'][-1],
    })


print(pd.DataFrame(data).to_string(index=False))

       Model  Train Accuracy  Val Accuracy
    Baseline        0.889729      0.871083
Experiment 1        0.900229      0.877917
Experiment 2        0.893792      0.874167
Experiment 3        0.846917      0.840167


**Experiment 1** has the highest validation accuracy with an accuracy of **87.8%**

In [12]:
print("Test accuracy:", exp_1_model.evaluate(x_test, y_test, verbose=0)[1])

Test accuracy: 0.8648999929428101
